# **Preparing the Silver Layer**

In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 3, Finished, Available, Finished, False)

# Clean Customers Table

In [5]:
customers = spark.table("customers")
display(customers)


StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eac2a76b-3651-4e6b-9c08-48ccf97b64d4)

In [14]:
cst_silver = (
    customers

    # Clean Email
    .withColumn(
        "Email",
        lower(trim(col("EMAIL")))
    )

    # Clean Name
    .withColumn(
        "Name",
        initcap(trim(col("name")))
    )

    # Standardize Gender
    .withColumn(
        "Gender",
        when(lower(trim(col("gender"))).isin("f", "female"), "Female")
        .when(lower(trim(col("gender"))).isin("m", "male"), "Male")
        .otherwise(initcap(trim(col("gender"))))
    )

    # Clean DOB separators
    .withColumn(
        "clean_dob",
        regexp_replace(col("dob"), "/", "-")
    )

    # Parse DOB formats
    .withColumn(
        "parsed_dob",
        coalesce(
            to_date(col("clean_dob"), "yyyy-MM-dd"),
            to_date(col("clean_dob"), "dd-MM-yyyy"),
            to_date(col("clean_dob"), "yyyyMMdd")
        )
    )

    # Replace invalid DOB
    .withColumn(
        "DOB",
        date_format(
            coalesce(
                col("parsed_dob"),
                lit("2099-01-01").cast("date")
            ),
            "yyyy-MM-dd"
        )
    )

    # Clean Location
    .withColumn(
        "Location",
        initcap(trim(col("location")))
    )

    # Duplicate flag
    .withColumn(
        "Flag_Duplicates",
        row_number().over(
            Window.partitionBy("customer_id").orderBy("customer_id")
        )
    )

    # Drop temp columns
    .drop("clean_dob", "parsed_dob")
)

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 16, Finished, Available, Finished, False)

In [15]:
display(cst_silver)

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6e8aaa39-922c-4701-aee5-1fd85ba9da04)

In [8]:
cst_silver.write.format("delta").mode("overwrite").saveAsTable("cst_silver")


StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 10, Finished, Available, Finished, False)

# Clean Orders Table

In [10]:
orders= spark.table("orders")

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 12, Finished, Available, Finished, False)

In [11]:
display(orders)

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8302c62e-6bbd-4fce-ac9a-070d903a7aab)

In [12]:
orders_silver = (
    orders

    # Standardize date separators first (- → /)
    .withColumn(
        "clean_date",
        regexp_replace(col("order_date"), "-", "/")
    )

    # Convert mixed formats into true DateType
    .withColumn(
        "parsed_date",
        coalesce(
            to_date(col("clean_date"), "yyyy/MM/dd"),  # 2022/07/15
            to_date(col("clean_date"), "dd/MM/yyyy"),  # 15/07/2022
            to_date(col("clean_date"), "yyyyMMdd"),    # 20220715
            to_date(col("clean_date"), "ddMMyyyy")     # 15072022
        )
    )

    # Final standard format: YYYY/MM/DD
    .withColumn(
        "Order_Date",
        date_format(col("parsed_date"), "yyyy/MM/dd")
    )

    # Clean amount
    .withColumn(
        "Amount",
        coalesce(
            col("amount").cast(DoubleType()),
            lit(0.0)
        )
    )

    # Clean status
    .withColumn(
        "Status",
        initcap(col("status"))
    )

    # Duplicate count order (1 = first, 2 = second duplicate, 3 = third...)
    .withColumn(
        "Flag_Duplicates",
        row_number().over(
            Window.partitionBy("order_id").orderBy("order_id")
        )
    )

    # Drop temp columns
    .drop("clean_date", "parsed_date")
)

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 14, Finished, Available, Finished, False)

In [13]:
display(orders_silver)

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 07db7f82-51b5-4b25-bebd-288a9d21a331)

In [16]:
orders_silver.write.format("delta").mode("overwrite").saveAsTable("orders_silver")


StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 18, Finished, Available, Finished, False)

# Clean Support_Tickets Table

In [18]:
support_tickets = spark.table("support_tickets")

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 20, Finished, Available, Finished, False)

In [19]:
display(support_tickets)

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5dac0713-e48e-44b1-8fed-ba4a72a87597)

In [20]:
support_tickets_silver = (
    support_tickets

    # Clean Issue Type
    .withColumn(
        "Issue_Type",
        initcap(col("issue_type"))
    )
    .withColumn(
        "Issue_Type",
        when(col("Issue_Type") == "Na", "No Issue")
        .otherwise(col("Issue_Type"))
    )

    # Replace / with -
    .withColumn(
        "clean_date",
        regexp_replace(col("ticket_date"), "/", "-")
    )

    # Replace n/a with null
    .withColumn(
        "clean_date",
        when(col("clean_date") == "n/a", None)
        .otherwise(col("clean_date"))
    )

    # Parse all formats
    .withColumn(
        "parsed_date",
        coalesce(
            to_date(col("clean_date"), "yyyy-MM-dd"),  # 2022-07-19
            to_date(col("clean_date"), "dd-MM-yyyy"),  # 19-07-2022
            to_date(col("clean_date"), "yyyyMMdd")     # 20220722
        )
    )

    # Replace invalid/null with default date
    .withColumn(
        "parsed_date",
        coalesce(
            col("parsed_date"),
            lit("2099-01-01").cast("date")
        )
    )

    # Final format yyyy-MM-dd
    .withColumn(
        "Ticket_Date",
        date_format(col("parsed_date"), "yyyy-MM-dd")
    )

    # Clean Resolution Status
    .withColumn(
        "Resolution_Status",
        coalesce(
            initcap(col("resolution_status")),
            lit("Unknown")
        )
    )

    # Duplicate counter (1,2,3...)
    .withColumn(
        "Flag_Duplicates",
        row_number().over(
            Window.partitionBy("ticket_id").orderBy("ticket_id")
        )
    )

    # Drop temp columns
    .drop("clean_date", "parsed_date")
)

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 22, Finished, Available, Finished, False)

In [21]:
display(support_tickets_silver)

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c719f18e-7a72-4744-a6cd-29f04c9fd106)

In [22]:
support_tickets_silver.write.format('delta').mode("overwrite").saveAsTable("support_tickets_silver")

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 24, Finished, Available, Finished, False)

# Clean Payments Table

In [23]:
payments = spark.table("payments")

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 25, Finished, Available, Finished, False)

In [24]:
display(payments)

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5dc44ecc-2a95-4779-8cfb-e53c454aa714)

In [25]:
payments_silver = (
    payments

    # Replace / with -
    .withColumn(
        "clean_date",
        regexp_replace(col("payment_date"), "/", "-")
    )

    # Replace n/a with null
    .withColumn(
        "clean_date",
        when(col("clean_date") == "n/a", None)
        .otherwise(col("clean_date"))
    )

    # Parse all formats
    .withColumn(
        "parsed_date",
        coalesce(
            to_date(col("clean_date"), "yyyy-MM-dd"),  # 2022-07-16
            to_date(col("clean_date"), "dd-MM-yyyy"),  # 17-07-2022
            to_date(col("clean_date"), "yyyyMMdd")     # 20220722
        )
    )

    # Default invalid/null dates
    .withColumn(
        "parsed_date",
        coalesce(
            col("parsed_date"),
            lit("2099-01-01").cast("date")
        )
    )

    # Final date format
    .withColumn(
        "Payment_Date",
        date_format(col("parsed_date"), "yyyy-MM-dd")
    )

    # Clean payment method
    .withColumn(
        "Payment_Method",
        initcap(col("payment_method"))
    )

    # Clean payment status + replace null with Unknown
    .withColumn(
        "Payment_Status",
        coalesce(
            initcap(col("payment_status")),
            lit("Unknown")
        )
    )

    # Clean amount + replace null with 0
    .withColumn(
        "Amount",
        coalesce(
            col("amount").cast(DoubleType()),
            lit(0.0)
        )
    )

    # Duplicate counter (1,2,3...)
    .withColumn(
        "Flag_Duplicates",
        row_number().over(
            Window.partitionBy("payment_id").orderBy("payment_id")
        )
    )

    # Drop temp columns
    .drop("clean_date", "parsed_date")
)


StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 27, Finished, Available, Finished, False)

In [26]:
display(payments_silver)

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 82a12404-3656-4d24-b14e-b0a6d6d8db46)

In [27]:
payments_silver.write.format("delta").mode("overwrite").saveAsTable("payments_silver")

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 29, Finished, Available, Finished, False)

# Clean Web Activities Table

In [28]:
web_activities = spark.table("web_activities")

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 30, Finished, Available, Finished, False)

In [29]:
display(web_activities)

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c5a5647e-1ef2-4582-9ace-0d72f2a5cbdc)

In [30]:
web_activities_silver = (
    web_activities

    # Clean Page Viewed (replace / with space, then proper case)
    .withColumn(
        "Page_Viewed",
        initcap(
            regexp_replace(col("page_viewed"), "/", " ")
        )
    )

    # Replace / with -
    .withColumn(
        "clean_date",
        regexp_replace(col("session_time"), "/", "-")
    )

    # Replace n/a with null
    .withColumn(
        "clean_date",
        when(col("clean_date") == "n/a", None)
        .otherwise(col("clean_date"))
    )

    # Parse all date formats
    .withColumn(
        "parsed_date",
        coalesce(
            to_date(col("clean_date"), "yyyy-MM-dd"),  # 2022-07-18
            to_date(col("clean_date"), "dd-MM-yyyy"),  # 18-07-2022
            to_date(col("clean_date"), "yyyyMMdd")     # 20220719
        )
    )

    # Replace invalid/null with default date
    .withColumn(
        "parsed_date",
        coalesce(
            col("parsed_date"),
            lit("2099-01-01").cast("date")
        )
    )

    # Final standard format
    .withColumn(
        "Session_Time",
        date_format(col("parsed_date"), "yyyy-MM-dd")
    )

    # Duplicate counter
    .withColumn(
        "Flag_Duplicates",
        row_number().over(
            Window.partitionBy("session_id").orderBy("session_id")
        )
    )

    # Drop temp columns
    .drop("clean_date", "parsed_date")
    .withColumn("Device_Type",initcap(col="device_type"))
)

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 32, Finished, Available, Finished, False)

In [31]:
display(web_activities_silver)

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f2b21954-ca41-4a87-bad7-a83cc6497914)

In [32]:
web_activities_silver.write.format("delta").mode("overwrite").saveAsTable("web_activities_silver")

StatementMeta(, 9dcef010-dd23-43be-96a8-bdabf955de23, 34, Finished, Available, Finished, False)